# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [ ]:
import os, zipfile, json, sqlite3, time, subprocess, sys
from pathlib import Path
import pandas as pd, requests
import matplotlib.pyplot as plt

# Local run: robust path detection
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif (Path.cwd() / "HW_2" / "flasheats-classroom-pack" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "HW_2" / "flasheats-classroom-pack"
elif (Path.cwd().parent / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent
elif Path("/content/flasheats_classroom/database/flasheats.db").exists():
    BASE = Path("/content/flasheats_classroom")
else:
    for candidate in [Path.cwd(), Path.cwd().parent, Path("/content")]:
        for match in candidate.rglob("flasheats.db"):
            BASE = match.parent.parent
            break
        if BASE is not None:
            break

if BASE is None or not (BASE / 'database' / 'flasheats.db').exists():
    raise FileNotFoundError("flasheats.db not found. Check repository layout.")

print("Pack root located at:", BASE)


## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

In [ ]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

In [ ]:
# Deduplicate orders by order_id and analyze delivery outcomes
orders_df = pd.read_sql("SELECT * FROM orders;", con)
orders_df = orders_df.drop_duplicates(subset=["order_id"], keep="first").copy()

for col in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders_df[col] = pd.to_datetime(orders_df[col], format="mixed", errors="coerce")

orders_df["final_status_norm"] = orders_df["final_status"].astype(str).str.strip().str.lower()
delivered = orders_df[(orders_df["final_status_norm"] == "delivered") & orders_df["actual_delivery_at"].notna()].copy()
delivered["delay_min"] = (delivered["actual_delivery_at"] - delivered["promised_eta"]).dt.total_seconds() / 60.0

late_orders = delivered[delivered["delay_min"] > 0.0]
late_10m = delivered[delivered["delay_min"] > 10.0]

print(f"Total unique orders: {len(orders_df)}")
print(f"Delivered orders with timestamps: {len(delivered)}")
print(f"Late orders (> 0 min): {len(late_orders)} ({len(late_orders) / len(delivered):.2%})")
print(f"Meaningfully late orders (> 10 min): {len(late_10m)} ({len(late_10m) / len(delivered):.2%})")
print(f"Median delay among late orders: {late_orders['delay_min'].median():.2f} min")

# Display sample worst delays
display(delivered.sort_values(by="delay_min", ascending=False)[["order_id", "promised_eta", "actual_delivery_at", "delay_min"]].head(5))


### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [ ]:
# Multi-dimensional slice analysis: Traffic, Weather, Distance
def slice_summary(df_sub):
    n = len(df_sub)
    med = df_sub["delay_min"].median()
    late_cnt = (df_sub["delay_min"] > 0).sum()
    return pd.Series({"count": n, "median_delay": round(med, 2), "late_rate": round(late_cnt / n if n > 0 else 0, 4)})

traffic_stats = delivered.groupby("traffic_bucket").apply(slice_summary).reset_index()
print("Traffic breakdown:")
display(traffic_stats)

weather_stats = delivered.groupby("weather_bucket").apply(slice_summary).reset_index()
print("\nWeather breakdown:")
display(weather_stats)

dist_corr = delivered["distance_km_estimate"].corr(delivered["delay_min"])
print(f"\nDistance vs Delay correlation: {dist_corr:.4f}")


## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [ ]:
tickets = pd.read_csv(BASE / 'data' / 'support_tickets.csv')
print(f"Total support tickets: {len(tickets)}")
print(f"Duplicate tickets: {tickets['ticket_id'].duplicated().sum()}")
print(f"Null order IDs: {tickets['order_id'].isna().sum()}")

# Normalize categories
tickets_clean = tickets.copy()
tickets_clean["category_norm"] = tickets_clean["category"].astype(str).str.strip().str.lower()
tickets_clean["category_norm"] = tickets_clean["category_norm"].replace({"late delivery": "late_delivery", "eta issue": "eta_changed"})

print("\nComplaint category breakdown:")
cat_breakdown = pd.DataFrame({
    "ticket_count": tickets_clean["category_norm"].value_counts(),
    "share_pct": (tickets_clean["category_norm"].value_counts(normalize=True) * 100).round(2)
})
display(cat_breakdown)


## Challenge 4 — Retrieve Dispatch data reliably

In [ ]:
api_proc=subprocess.Popen([sys.executable,str(BASE/'api'/'mock_dispatch_api.py')],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
requests.get('http://127.0.0.1:8000/health').json()

In [ ]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [ ]:
RAW_DIR = BASE / 'student_output' / 'raw_dispatch'
RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_dispatch_orders():
    url = 'http://127.0.0.1:8000/dispatch/orders'
    page = 1
    page_size = 50
    records = []
    max_retries = 4

    while True:
        success = False
        attempts = 0

        while attempts < max_retries and not success:
            attempts += 1
            try:
                r = requests.get(url, params={'page': page, 'page_size': page_size}, timeout=10)
                if r.status_code == 200:
                    payload = r.json()
                    raw_file = RAW_DIR / f'page_{page:03d}.json'
                    with open(raw_file, 'w', encoding='utf-8') as fp:
                        json.dump(payload, fp)
                    records.extend(payload.get('data', []))
                    success = True
                    if not payload.get('has_more', False):
                        return records
                elif r.status_code == 429:
                    wait_time = r.json().get('retry_after_seconds', 1)
                    time.sleep(wait_time + 0.1)
                elif r.status_code == 500:
                    time.sleep(0.5 * attempts)
                else:
                    r.raise_for_status()
            except requests.RequestException:
                time.sleep(0.5 * attempts)

        if not success:
            raise RuntimeError(f'Failed on page {page}')
        page += 1

dispatch_records = fetch_all_dispatch_orders()
print(f"Retrieved {len(dispatch_records)} dispatch orders across {len(list(RAW_DIR.glob('page_*.json')))} raw page files.")


## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [ ]:
with open(BASE / 'data' / 'driver_events.json') as f:
    driver_events = json.load(f)

flat_events = []
for driver in driver_events:
    d_id = driver["driver_id"]
    for ev in driver["events"]:
        entry = {"driver_id": d_id}
        for k, v in ev.items():
            entry[k] = v
        flat_events.append(entry)

driver_df = pd.DataFrame(flat_events)
print("Total driver events:", len(driver_df))
print("\nEvent types breakdown:")
display(driver_df["type"].value_counts())

arrived_cnt = (driver_df["type"].astype(str).str.lower() == "driver_arrived").sum()
print(f"Explicit driver_arrived events found: {arrived_cnt}")


## Final FDE recommendation

### Executive Briefing: Problem Sizing & Decision Gate

1. **Problem Size:** Late delivery rate is **56.39%** ($843 / 1,495$), but drops to **23.34%** for significant delays ($>10$ min).
2. **Evidence:** Traffic accounts for only an **8.3%** late rate variance; road distance correlation is near zero ($r = 0.010$). Complaints cluster around kitchen prep and unstable ETAs.
3. **Trusted Sources:** SQLite orders for contractual delivery times; dispatch API for driver assignment; support tickets for customer friction.
4. **Uncertainty:** Inability to decouple courier transit time from kitchen wait time due to missing restaurant arrival timestamps.
5. **Instrumentation:** Mandate merchant "Order Ready" confirmation and automated driver geofence arrival telemetry.
6. **Decision: DO NOT build the AI delay predictor now.** Fix source instrumentation and dispatch timing first.
